In [ ]:
# 1. Verifikasi Lingkungan GPU & Pustaka Pendukung
!nvidia-smi
!pip install -q timm thop scikit-learn pandas matplotlib seaborn

# Pelatihan & Benchmark Komparasi Arsitektur FLAT 1-Stage (16 Kelas) — Official Complete

Dokumen notebook resmi ini memuat pipeline **FLAT 1-Stage (16 Kelas Gabungan)** untuk melatih dan mengevaluasi secara langsung (*real training, fair comparison*) arsitektur Convolutional Neural Networks (CNN) lintas 7 backbone lengkap:
1. `convnext_tiny` (**ConvNeXt-Tiny — Baseline**)
2. `resnet50` (**ResNet-50**)
3. `resnet101` (**ResNet-101**)
4. `efficientnet_b0` (**EfficientNet-B0**)
5. `efficientnetv2_rw_m` (**EfficientNetV2-M**)
6. `mobilenetv3_large_100` (**MobileNetV3-Large**)
7. `densenet121` (**DenseNet-121**)

*(Tersedia opsi pemilihan subset model pada konfigurasi `RUN_BACKBONES` untuk penyesuaian kuota komputasi GPU Kaggle / Google Colab).*

### Protokol Eksperimen Terstandarisasi:
1. **Target Klasifikasi**: 16 Kelas Gabungan secara langsung (*Single-Stage 16-Class Flat*).
2. **Validasi**: Stratified 5-Fold Cross Validation penuh (Fold 1 s.d. 5).
3. **Optimasi**: 100 Epochs per fold dengan 2-Phase Warmup (8 Epochs Freeze Backbone, dilanjutkan Discriminative Learning Rate: Backbone `3e-5`, Head `2.5e-4`, Cosine Annealing, dan Early Stopping Patience `15`).
4. **Standardisasi Classifier Head**: Seluruh backbone diintegrasikan dengan arsitektur head identik: `Linear(in_features, 512) -> LayerNorm(512) -> GELU() -> Dropout(0.3) -> Linear(512, 16)`.

## 1. Import Libraries, Random Seed & Pengaturan Direktori Lingkungan

In [ ]:
import os
import sys
import json
import time
import shutil
import random
import zipfile
import urllib.request
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.optim.lr_scheduler import CosineAnnealingLR

import timm
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

try:
    from thop import profile as thop_profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False

# --- REPRODUCIBILITY SEED ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# --- DEVICE ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[INFO] Running on Device: {device}')
if torch.cuda.is_available():
    print(f'[INFO] GPU: {torch.cuda.get_device_name(0)}')

# --- ENVIRONMENT & WORKING DIRECTORIES (Kaggle & Colab Auto-Detection) ---
if os.path.exists('/kaggle/working'):
    WORK_DIR = '/kaggle/working'
    INPUT_BASE = '/kaggle/input'
elif os.path.exists('/content'):
    WORK_DIR = '/content'
    INPUT_BASE = '/content'
else:
    WORK_DIR = os.getcwd()
    INPUT_BASE = os.getcwd()

HF_DATASET_URL = 'https://huggingface.co/turasa22/klasifikasibijikopiconvnext/resolve/main/Dataset_Split2.zip'
TARGET_SPLIT2_DIR = None

candidate_paths = [
    os.path.join(WORK_DIR, 'Dataset_Split2'),
    os.path.join(INPUT_BASE, 'Dataset_Split2'),
    os.path.join(INPUT_BASE, 'dataset-split2'),
    os.path.join(INPUT_BASE, 'coffee-beans-dataset-split2'),
]
for p in candidate_paths:
    if os.path.isdir(p) and os.path.exists(os.path.join(p, 'fold_1')):
        TARGET_SPLIT2_DIR = os.path.abspath(p)
        break

if TARGET_SPLIT2_DIR is None:
    for root, dirs, files in os.walk(INPUT_BASE):
        if 'fold_1' in dirs and 'fold_2' in dirs:
            TARGET_SPLIT2_DIR = root
            break

if TARGET_SPLIT2_DIR is None:
    found_zip = None
    for root, dirs, files in os.walk(INPUT_BASE):
        for f in files:
            if f.lower().endswith('.zip') and 'split2' in f.lower():
                found_zip = os.path.join(root, f)
                break
        if found_zip:
            break
            
    if found_zip:
        print(f'[INFO] Mengekstrak dataset dari Kaggle Input: {found_zip}')
        with zipfile.ZipFile(found_zip, 'r') as zf:
            zf.extractall(WORK_DIR)
    else:
        zip_dest = os.path.join(WORK_DIR, 'Dataset_Split2.zip')
        print(f'[INFO] Mengunduh dataset resmi dari Hugging Face ({HF_DATASET_URL})...')
        urllib.request.urlretrieve(HF_DATASET_URL, zip_dest)
        with zipfile.ZipFile(zip_dest, 'r') as zf:
            zf.extractall(WORK_DIR)
            
    for root, dirs, files in os.walk(WORK_DIR):
        if 'fold_1' in dirs and 'fold_2' in dirs:
            TARGET_SPLIT2_DIR = root
            break

MODELS_BASE_DIR = os.path.join(WORK_DIR, 'models', 'FLAT')
REPORTS_BASE_DIR = os.path.join(WORK_DIR, 'reports', 'FLAT')
os.makedirs(MODELS_BASE_DIR, exist_ok=True)
os.makedirs(REPORTS_BASE_DIR, exist_ok=True)

print(f'[INFO] Direktori Dataset: {TARGET_SPLIT2_DIR}')
print(f'[INFO] Simpan Model Ke  : {MODELS_BASE_DIR}')
print(f'[INFO] Simpan Report Ke : {REPORTS_BASE_DIR}')

## 2. Pemetaan Label 16 Kelas & Verifikasi Partisi Fold

In [ ]:
# --- DEFINISI LABEL RESMI (16 KELAS FLAT: SPESIES x SANGRAI) ---
JENIS_LABELS   = {'arabika': 0, 'ekselsa': 1, 'liberika': 2, 'robusta': 3}
SANGRAI_LABELS = {'dark': 0, 'green': 1, 'light': 2, 'medium': 3}

SPECIES_NAMES = ['Arabika', 'Ekselsa', 'Liberika', 'Robusta']
ROAST_NAMES   = ['Dark', 'Green', 'Light', 'Medium']

CLASS_NAMES_16 = [
    f'{j}_{s}'
    for j in SPECIES_NAMES
    for s in ROAST_NAMES
]

class CoffeeDatasetFlatFolder(Dataset):
    """
    Dataset kustom untuk membaca struktur folder Dataset_Split2/fold_X/train atau val
    berdasarkan 16 kelas gabungan (Jenis x Sangrai).
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.data = []

        for j_folder in sorted(os.listdir(root_dir)):
            j_path = os.path.join(root_dir, j_folder)
            if not os.path.isdir(j_path):
                continue
            j_idx = JENIS_LABELS.get(j_folder.lower(), -1)
            if j_idx == -1:
                continue

            for s_folder in sorted(os.listdir(j_path)):
                s_path = os.path.join(j_path, s_folder)
                if not os.path.isdir(s_path):
                    continue
                s_idx = SANGRAI_LABELS.get(s_folder.lower(), -1)
                if s_idx == -1:
                    continue

                lbl_flat = (j_idx * 4) + s_idx
                for img_name in sorted(os.listdir(s_path)):
                    if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                        self.data.append((os.path.join(s_path, img_name), lbl_flat))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data[idx]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

print(f'[INFO] Status Ketersediaan Fold di: {TARGET_SPLIT2_DIR}')
if TARGET_SPLIT2_DIR:
    for fold_idx in range(1, 6):
        f_tr = os.path.join(TARGET_SPLIT2_DIR, f'fold_{fold_idx}', 'train')
        f_va = os.path.join(TARGET_SPLIT2_DIR, f'fold_{fold_idx}', 'val')
        ds_tr = CoffeeDatasetFlatFolder(f_tr) if os.path.exists(f_tr) else []
        ds_va = CoffeeDatasetFlatFolder(f_va) if os.path.exists(f_va) else []
        print(f'  Fold {fold_idx}: Train = {len(ds_tr)} citra | Val = {len(ds_va)} citra (Total = {len(ds_tr) + len(ds_va)})')

## 3. Augmentasi Citra & Arsitektur Backbone Modular Universal (16 Kelas)
### Penjelasan Desain Arsitektur & Perbedaan Teknis:
1. **Universal Classifier Head (16 Kelas)**:
   Setiap arsitektur memiliki representasi dimensi laten akhir `in_features` yang berbeda (ConvNeXt-Tiny: 768, ResNet-50: 2048, ResNet-101: 2048, EfficientNet-B0: 1280, EfficientNetV2-M: 2152, MobileNetV3-Large: 1280, DenseNet-121: 1024).
   Untuk perbandingan yang adil (*fair comparison*), classification head bawaan digantikan dengan arsitektur seragam:
   $$\text{Head} = \text{Linear}(in\_features, 512) \rightarrow \text{LayerNorm}(512) \rightarrow \text{GELU}() \rightarrow \text{Dropout}(0.3) \rightarrow \text{Linear}(512, 16)$$
2. **Augmentasi Terukur (*Mild Regularization*)**:
   Menggunakan `RandomHorizontalFlip(0.5)`, `RandomVerticalFlip(0.5)`, `RandomRotation(15)`, dan `ColorJitter(0.05, 0.05)` yang identik dengan protokol resmi riset.

In [ ]:
def get_transforms(img_size=224):
    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.05, contrast=0.05),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    val_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return train_tf, val_tf

# --- REGISTRI ARSITEKTUR BACKBONE ---
BACKBONE_REGISTRY = {
    'convnext_tiny': {
        'name': 'ConvNeXt-Tiny (Baseline)',
        'family': 'ConvNeXt',
        'timm_name': 'convnext_tiny',
        'desc': 'Modern pure CNN baseline incorporating Vision Transformer design.'
    },
    'resnet50': {
        'name': 'ResNet-50',
        'family': 'ResNet',
        'timm_name': 'resnet50',
        'desc': 'Residual learning 50 layers with bottleneck skip-connections.'
    },
    'resnet101': {
        'name': 'ResNet-101',
        'family': 'ResNet',
        'timm_name': 'resnet101',
        'desc': 'Deeper residual network (101 layers) for higher feature capacity.'
    },
    'efficientnet_b0': {
        'name': 'EfficientNet-B0',
        'family': 'EfficientNet V1',
        'timm_name': 'efficientnet_b0',
        'desc': 'Compound scaling depth, width, and resolution (lightweight).'
    },
    'efficientnetv2_rw_m': {
        'name': 'EfficientNetV2-M',
        'family': 'EfficientNet V2',
        'timm_name': 'efficientnetv2_rw_m',
        'desc': 'Fused-MBConv architecture with improved training speed and accuracy.'
    },
    'mobilenetv3_large_100': {
        'name': 'MobileNetV3-Large',
        'family': 'MobileNet',
        'timm_name': 'mobilenetv3_large_100',
        'desc': 'Mobile/Edge optimized CNN with hard-swish and squeeze-and-excitation.'
    },
    'densenet121': {
        'name': 'DenseNet-121',
        'family': 'DenseNet',
        'timm_name': 'densenet121',
        'desc': 'Dense feature connectivity with direct feed-forward reuse.'
    }
}

class UniversalFlatClassifier(nn.Module):
    def __init__(self, backbone_key='convnext_tiny', num_classes=16, drop_rate=0.3, pretrained=True):
        super().__init__()
        self.backbone_key = backbone_key
        cfg = BACKBONE_REGISTRY[backbone_key]
        self.backbone = timm.create_model(cfg['timm_name'], pretrained=pretrained)
        in_features = self.backbone.get_classifier().in_features
        self.backbone.reset_classifier(0)  # Reset head bawaan ke identity

        # Standarisasi classifier head sesuai protokol resmi (512 -> 16 kelas)
        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(drop_rate),
            nn.Linear(512, num_classes)
        )
        self.in_features = in_features

    def forward(self, x):
        feat = self.backbone(x)
        return self.head(feat)

print('Verifikasi Instansiasi Head 16-Kelas FLAT untuk Setiap Arsitektur:')
for k, v in BACKBONE_REGISTRY.items():
    m = UniversalFlatClassifier(backbone_key=k, num_classes=16, pretrained=False)
    p_cnt = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"  [{k:22s}] in_features: {m.in_features:4d} -> Head: 512 -> 16 Kelas | Params: {p_cnt:.2f}M | {v['name']}")

## 4. Engine Pelatihan (2-Phase Warmup, Differential Learning Rate & Cosine Annealing)
### Rincian Strategi Optimasi:
- **Fase 1 (Warmup: Epoch 1 s.d. 8)**: Backbone dibekukan (*frozen*), hanya classification head yang dilatih dengan $\text{LR} = 5 \times 10^{-4}$ menggunakan `AdamW` (weight decay $0.01$).
- **Fase 2 (Unfreeze: Epoch 9 s.d. 100)**: Seluruh parameter model dibuka (*unfrozen*) dengan *Differential Learning Rate*:
  - Parameter Backbone: $\text{LR} = 3 \times 10^{-5}$
  - Parameter Head: $\text{LR} = 2.5 \times 10^{-4}$
  - Learning rate meluruh secara halus mengikuti jadwal `CosineAnnealingLR` hingga $\eta_{\min} = 10^{-6}$.
- **Regularisasi & Early Stopping**: `CrossEntropyLoss(label_smoothing=0.1)`, gradient norm clipping ($1.0$), dan early stopping dengan *patience* 15 epoch.

In [ ]:
def train_kfold_model(
    backbone_key,
    train_loader,
    val_loader,
    num_classes=16,
    save_name='model.pth',
    label_info='Fold 1',
    target_name=None,
    history_save_dir=None,
    num_epochs=100,
    warmup_epochs=8,
    patience=15
):
    model = UniversalFlatClassifier(backbone_key=backbone_key, num_classes=num_classes, pretrained=True).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    display_target = target_name if target_name is not None else label_info

    # --- FASE 1: WARMUP (Freeze Backbone) ---
    for p in model.backbone.parameters():
        p.requires_grad = False
    for p in model.head.parameters():
        p.requires_grad = True

    optimizer = optim.AdamW(
        model.head.parameters(),
        lr=5e-4, weight_decay=1e-2
    )
    scheduler = None

    best_acc = 0.0
    best_val_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    print(f'\n[INFO] Device: {device} | Backbone: {BACKBONE_REGISTRY[backbone_key]["name"]} | Target: {display_target}')
    print(f'{"="*70}')
    print(f'  MULAI TRAINING: {label_info} | Output: {num_classes} Kelas | Max Epoch: {num_epochs}')
    print(f'{"="*70}')

    for epoch in range(num_epochs):
        # --- FASE 2: UNFREEZE (Differential Learning Rate) ---
        if epoch == warmup_epochs:
            for p in model.backbone.parameters():
                p.requires_grad = True

            optimizer = optim.AdamW([
                {'params': model.backbone.parameters(), 'lr': 3.0e-5},
                {'params': model.head.parameters(),     'lr': 2.5e-4},
            ], weight_decay=1e-2)

            scheduler = CosineAnnealingLR(optimizer, T_max=(num_epochs - warmup_epochs), eta_min=1e-6)

        # Training Phase
        model.train()
        running_loss, train_correct, train_total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss  += loss.item() * inputs.size(0)
            _, preds       = torch.max(outputs, 1)
            train_total   += labels.size(0)
            train_correct += (preds == labels).sum().item()

        if epoch >= warmup_epochs and scheduler is not None:
            scheduler.step()

        epoch_tr_loss = running_loss / len(train_loader.dataset)
        epoch_tr_acc  = train_correct / train_total

        # Validation Phase
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss    = criterion(outputs, labels)

                val_loss    += loss.item() * inputs.size(0)
                _, preds     = torch.max(outputs, 1)
                val_total   += labels.size(0)
                val_correct += (preds == labels).sum().item()

        epoch_va_loss = val_loss / len(val_loader.dataset)
        epoch_va_acc  = val_correct / val_total

        history['train_loss'].append(epoch_tr_loss)
        history['train_acc'].append(epoch_tr_acc)
        history['val_loss'].append(epoch_va_loss)
        history['val_acc'].append(epoch_va_acc)

        if (epoch + 1) % 5 == 0 or epoch == 0 or epoch == num_epochs - 1:
            print(f"{label_info} | Epoch {epoch+1:02d}/{num_epochs:02d} | Train Loss: {epoch_tr_loss:.4f} Acc: {epoch_tr_acc*100:.2f}% | Val Loss: {epoch_va_loss:.4f} Acc: {epoch_va_acc*100:.2f}%")

        if epoch_va_acc > best_acc:
            best_acc = epoch_va_acc
            torch.save(model.state_dict(), save_name)

        if epoch_va_loss < best_val_loss:
            best_val_loss = epoch_va_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience and epoch >= warmup_epochs + 7:
                print(f"[Early Stopping] {label_info} berhenti pada epoch {epoch+1}. Best Val Acc: {best_acc*100:.2f}%")
                break

    print(f"=> {label_info} Selesai! Best Val Accuracy: {best_acc*100:.2f}%")

    # Simpan Grafik Loss & Accuracy
    if history_save_dir is not None:
        os.makedirs(history_save_dir, exist_ok=True)
        epochs_range = range(1, len(history['train_loss']) + 1)
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 2, 1)
        plt.plot(epochs_range, history['train_loss'], label='Train Loss')
        plt.plot(epochs_range, history['val_loss'],   label='Val Loss')
        plt.legend()
        plt.title(f"{BACKBONE_REGISTRY[backbone_key]['name']} — Loss ({display_target})")
        plt.grid(True)

        plt.subplot(1, 2, 2)
        plt.plot(epochs_range, history['train_acc'], label='Train Acc')
        plt.plot(epochs_range, history['val_acc'],   label='Val Acc')
        plt.legend()
        plt.title(f"{BACKBONE_REGISTRY[backbone_key]['name']} — Accuracy ({display_target})")
        plt.grid(True)
        plt.tight_layout()

        safe_name = label_info.replace(' ', '_').replace('/', '_')
        plot_path = os.path.join(history_save_dir, f"History_{safe_name}.png")
        try:
            plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        except Exception:
            pass
        plt.close()

    return history, best_acc

## 5. Eksekusi Utama: FLAT 5-Fold Cross Validation untuk Seluruh 7 Backbone

In [ ]:
# --- PENGATURAN EKSEKUSI BENCHMARK FLAT (All 7 Backbones) ---
RUN_BACKBONES = [
    'convnext_tiny',        # ConvNeXt-Tiny (Baseline)
    'resnet50',             # ResNet-50
    'resnet101',            # ResNet-101
    'efficientnet_b0',      # EfficientNet V1
    'efficientnetv2_rw_m',  # EfficientNet V2
    'mobilenetv3_large_100',# MobileNetV3
    'densenet121'           # DenseNet-121
]

FOLDS_TO_RUN  = range(1, 6)   # 5-Fold Lengkap (1 sampai 5)
NUM_EPOCHS    = 100          # 100 Epochs per fold
WARMUP_EPOCHS = 8            # 8 Epochs Warmup (freeze backbone)
PATIENCE      = 15           # Patience early stopping
BATCH_SIZE    = 32

train_tf, val_tf = get_transforms(img_size=224)
global_backbone_summary = []

for b_key in RUN_BACKBONES:
    b_info = BACKBONE_REGISTRY[b_key]
    print(f'\n{"="*85}')
    print(f'       MEMULAI FLAT 5-FOLD BENCHMARK: {b_info["name"]} ({b_info["family"]})')
    print(f'{"="*85}')

    b_models_dir = os.path.join(MODELS_BASE_DIR, b_key)
    b_reports_dir = os.path.join(REPORTS_BASE_DIR, b_key)
    b_hist_dir = os.path.join(b_reports_dir, 'Training_Histories')
    os.makedirs(b_models_dir, exist_ok=True)
    os.makedirs(b_reports_dir, exist_ok=True)
    os.makedirs(b_hist_dir, exist_ok=True)

    flat_kfold_metrics = []
    oof_flat_targets = []
    oof_flat_preds = []
    t_backbone_start = time.time()

    for fold_num in FOLDS_TO_RUN:
        fold_train_root = os.path.join(TARGET_SPLIT2_DIR, f'fold_{fold_num}', 'train')
        fold_val_root   = os.path.join(TARGET_SPLIT2_DIR, f'fold_{fold_num}', 'val')

        print(f'\n[FLAT {b_info["name"]}] --- FOLD {fold_num} ---')

        train_ds = CoffeeDatasetFlatFolder(fold_train_root, transform=train_tf)
        val_ds   = CoffeeDatasetFlatFolder(fold_val_root,   transform=val_tf)

        loader_tr = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
        loader_va = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

        m_flat_path = os.path.join(b_models_dir, f'{b_key}_FLAT_Fold{fold_num}.pth')
        history, best_acc = train_kfold_model(
            backbone_key=b_key,
            train_loader=loader_tr,
            val_loader=loader_va,
            num_classes=16,
            save_name=m_flat_path,
            label_info=f'{b_key} FLAT Fold {fold_num}',
            target_name='16 Kelas Gabungan',
            history_save_dir=b_hist_dir,
            num_epochs=NUM_EPOCHS,
            warmup_epochs=WARMUP_EPOCHS,
            patience=PATIENCE
        )

        # Evaluasi Validasi Fold
        eval_model = UniversalFlatClassifier(backbone_key=b_key, num_classes=16, pretrained=False).to(device)
        eval_model.load_state_dict(torch.load(m_flat_path, map_location=device))
        eval_model.eval()

        fold_targets, fold_preds = [], []
        with torch.no_grad():
            for inputs, targets in loader_va:
                inputs = inputs.to(device)
                outputs = eval_model(inputs)
                _, p = torch.max(outputs, 1)
                fold_targets.extend(targets.cpu().numpy())
                fold_preds.extend(p.cpu().numpy())

        oof_flat_targets.extend(fold_targets)
        oof_flat_preds.extend(fold_preds)

        f_acc = accuracy_score(fold_targets, fold_preds) * 100
        f_p   = precision_score(fold_targets, fold_preds, average='macro', zero_division=0) * 100
        f_r   = recall_score(fold_targets, fold_preds, average='macro', zero_division=0) * 100
        f_f1  = f1_score(fold_targets, fold_preds, average='macro', zero_division=0) * 100
        f_f1w = f1_score(fold_targets, fold_preds, average='weighted', zero_division=0) * 100

        print(f'  [Fold {fold_num} Selesai] Val Acc: {f_acc:.2f}% | F1-Macro: {f_f1:.2f}%')
        flat_kfold_metrics.append({
            'Fold': f'Fold {fold_num}',
            'Accuracy (%)': round(f_acc, 2),
            'Precision Macro (%)': round(f_p, 2),
            'Recall Macro (%)': round(f_r, 2),
            'F1 Macro (%)': round(f_f1, 2),
            'F1 Weighted (%)': round(f_f1w, 2)
        })

    t_backbone_total = time.time() - t_backbone_start

    # Hitung rata-rata 5-Fold
    df_kfold = pd.DataFrame(flat_kfold_metrics)
    mean_row = {
        'Fold': 'Mean ± Std',
        'Accuracy (%)': f"{df_kfold['Accuracy (%)'].mean():.2f} ± {df_kfold['Accuracy (%)'].std():.2f}",
        'Precision Macro (%)': f"{df_kfold['Precision Macro (%)'].mean():.2f} ± {df_kfold['Precision Macro (%)'].std():.2f}",
        'Recall Macro (%)': f"{df_kfold['Recall Macro (%)'].mean():.2f} ± {df_kfold['Recall Macro (%)'].std():.2f}",
        'F1 Macro (%)': f"{df_kfold['F1 Macro (%)'].mean():.2f} ± {df_kfold['F1 Macro (%)'].std():.2f}",
        'F1 Weighted (%)': f"{df_kfold['F1 Weighted (%)'].mean():.2f} ± {df_kfold['F1 Weighted (%)'].std():.2f}"
    }
    df_kfold = pd.concat([df_kfold, pd.DataFrame([mean_row])], ignore_index=True)
    kfold_csv = os.path.join(b_reports_dir, f'KFold_Validation_Metrics_FLAT_{b_key}.csv')
    df_kfold.to_csv(kfold_csv, index=False)
    print(f'[SAVED] Tabel 5-Fold Metrics disimpan di: {kfold_csv}')

    # Classification Report Out-of-Fold (OOF)
    report_dict = classification_report(oof_flat_targets, oof_flat_preds, target_names=CLASS_NAMES_16, output_dict=True, zero_division=0)
    df_report = pd.DataFrame(report_dict).transpose()
    df_report.to_csv(os.path.join(b_reports_dir, f'Classification_Report_OOF_FLAT_{b_key}.csv'))

    # Visualisasi OOF Confusion Matrix
    cm = confusion_matrix(oof_flat_targets, oof_flat_preds, normalize='true')
    plt.figure(figsize=(11, 9))
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASS_NAMES_16, yticklabels=CLASS_NAMES_16,
                cbar=True, annot_kws={"size": 8})
    plt.title(f'Normalized OOF Confusion Matrix 16 Kelas (FLAT)\n{b_info["name"]} (Mean Acc: {df_kfold.iloc[0:5]["Accuracy (%)"].astype(float).mean():.2f}%)', fontsize=12, pad=15)
    plt.xlabel('Predicted Label', fontsize=10)
    plt.ylabel('True Label', fontsize=10)
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    cm_path = os.path.join(b_reports_dir, f'KFold_ConfusionMatrix_OOF_FLAT_{b_key}.png')
    plt.savefig(cm_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f'[SAVED] Confusion matrix OOF disimpan di: {cm_path}')

    # Model Complexity Profiling (Params, GFLOPs, Latency, FPS)
    m_dummy = UniversalFlatClassifier(backbone_key=b_key, num_classes=16, pretrained=False).to(device)
    m_dummy.eval()
    dummy = torch.randn(1, 3, 224, 224, device=device)
    params_m = sum(p.numel() for p in m_dummy.parameters()) / 1e6
    gflops = 0.0
    if HAS_THOP:
        try:
            macs, _ = thop_profile(m_dummy, inputs=(dummy,), verbose=False)
            gflops = (macs * 2) / 1e9
        except Exception:
            pass

    with torch.no_grad():
        for _ in range(5):
            _ = m_dummy(dummy)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0_lat = time.perf_counter()
        for _ in range(50):
            _ = m_dummy(dummy)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1_lat = time.perf_counter()
    lat_ms = ((t1_lat - t0_lat) / 50) * 1000
    fps = 1000.0 / lat_ms if lat_ms > 0 else 0.0

    mean_acc = round(df_kfold.iloc[0:5]['Accuracy (%)'].astype(float).mean(), 2)
    mean_f1  = round(df_kfold.iloc[0:5]['F1 Macro (%)'].astype(float).mean(), 2)
    mean_p   = round(df_kfold.iloc[0:5]['Precision Macro (%)'].astype(float).mean(), 2)
    mean_r   = round(df_kfold.iloc[0:5]['Recall Macro (%)'].astype(float).mean(), 2)
    mean_f1w = round(df_kfold.iloc[0:5]['F1 Weighted (%)'].astype(float).mean(), 2)

    global_backbone_summary.append({
        'Backbone': b_key,
        'Model_Name': b_info['name'],
        'Family': b_info['family'],
        'Accuracy (%)': mean_acc,
        'F1_Macro (%)': mean_f1,
        'Precision_Macro (%)': mean_p,
        'Recall_Macro (%)': mean_r,
        'F1_Weighted (%)': mean_f1w,
        'Params_M': round(params_m, 2),
        'GFLOPs': round(gflops, 2),
        'Latency_ms': round(lat_ms, 2),
        'FPS': round(fps, 1),
        'Total_Train_Time_s': round(t_backbone_total, 1)
    })

    print(f'\n[SELESAI {b_info["name"]}] Mean Acc: {mean_acc}% | Mean F1 Macro: {mean_f1}% | Latency: {lat_ms:.2f} ms | FPS: {fps:.1f}')

## 6. Tabel Perbandingan Seluruh Backbone terhadap Baseline ConvNeXt-Tiny

In [ ]:
# --- TABEL SINTESIS KOMPARASI LINTAS 7 BACKBONE (FLAT COMPLETE) ---
df_all = pd.DataFrame(global_backbone_summary)

if 'convnext_tiny' in df_all['Backbone'].values:
    base_row = df_all[df_all['Backbone'] == 'convnext_tiny'].iloc[0]
    base_acc = base_row['Accuracy (%)']
    base_f1  = base_row['F1_Macro (%)']
    df_all['Delta_Acc_vs_Baseline (%)'] = df_all['Accuracy (%)'].apply(lambda x: round(x - base_acc, 2))
    df_all['Delta_F1_vs_Baseline (%)']  = df_all['F1_Macro (%)'].apply(lambda x: round(x - base_f1, 2))

df_all = df_all.sort_values(by='F1_Macro (%)', ascending=False).reset_index(drop=True)
df_all.index += 1
df_all.index.name = 'Rank'

print('\n' + '='*105)
print('       TABEL KOMPARASI LENGKAP 7 ARSITEKTUR BACKBONE (FLAT CLASSIFIER 16 KELAS)')
print('='*105)
show_cols = [
    'Model_Name', 'Family', 'Accuracy (%)', 'F1_Macro (%)',
    'Delta_Acc_vs_Baseline (%)', 'Delta_F1_vs_Baseline (%)',
    'Params_M', 'GFLOPs', 'Latency_ms', 'FPS', 'Total_Train_Time_s'
]
cols_to_print = [c for c in show_cols if c in df_all.columns]
print(df_all[cols_to_print].to_string())
print('='*105)

csv_path = os.path.join(REPORTS_BASE_DIR, 'Benchmark_FLAT_Backbones_Comparison.csv')
df_all.to_csv(csv_path)
print(f'[SAVED] Laporan CSV lengkap disimpan di: {csv_path}')

## 7. Visualisasi Hasil Komparasi & Trade-off Efisiensi

In [ ]:
# --- VISUALISASI PERBANDINGAN DAN FRONTIER EFISIENSI (FLAT COMPLETE) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

x_pos = np.arange(len(df_all))
width = 0.35

axes[0].bar(x_pos - width/2, df_all['Accuracy (%)'], width, label='16-Class Accuracy (%)', color='#1f77b4')
axes[0].bar(x_pos + width/2, df_all['F1_Macro (%)'], width, label='F1-Score Macro (%)', color='#ff7f0e')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(df_all['Model_Name'], rotation=30, ha='right', fontsize=10)
axes[0].set_ylim(min(df_all['F1_Macro (%)'].min() - 5, 80), 101)
axes[0].set_ylabel('Persentase (%)')
axes[0].set_title('Perbandingan Akurasi & F1-Score Macro Seluruh 7 Backbone (FLAT)')
axes[0].legend()
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

scatter = axes[1].scatter(
    df_all['GFLOPs'],
    df_all['F1_Macro (%)'],
    s=df_all['Params_M'] * 15,
    c=df_all['Latency_ms'],
    cmap='viridis',
    alpha=0.85,
    edgecolors='black',
    linewidth=1.5
)
cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label('Latency (ms)', fontsize=10)

for idx, row in df_all.iterrows():
    axes[1].annotate(
        row['Model_Name'],
        (row['GFLOPs'], row['F1_Macro (%)']),
        textcoords="offset points",
        xytext=(0, 10),
        ha='center',
        fontsize=9,
        fontweight='bold'
    )

axes[1].set_xlabel('GFLOPs (Inference Complexity)', fontsize=10)
axes[1].set_ylabel('F1-Score Macro (%)', fontsize=10)
axes[1].set_title('Trade-Off Komputasi vs Performa Klasifikasi Seluruh Backbone (FLAT)\n(Ukuran Bubble = Params M | Warna = Latensi ms)', fontsize=11)
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plot_comp_path = os.path.join(REPORTS_BASE_DIR, 'FLAT_All_Backbones_Visual_Comparison.png')
plt.savefig(plot_comp_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'[SAVED] Grafik komparasi lengkap disimpan di: {plot_comp_path}')

In [ ]:
# --- BUNDLING HASIL MODEL (.PTH) DAN LAPORAN KE ZIP (FLAT COMPLETE) ---
bundle_target = os.path.join(WORK_DIR, 'Hasil_FLAT_Backbones_Benchmark_Complete')
shutil.make_archive(bundle_target, 'zip', root_dir=WORK_DIR, base_dir='reports')
print(f'\n[SUCCESS] Seluruh laporan, CSV, dan grafik berhasil dikompres ke: {bundle_target}.zip')